# 🎬 HOTSTAR — AI Movie Recommendation System
### Data Driven Recommendations — Mini Project

---

| Component | Details |
|---|---|
| **Platform** | Disney+ Hotstar |
| **Algorithm** | Content-Based Filtering |
| **Technique** | TF-IDF Vectorization + Cosine Similarity |
| **Dataset** | TMDB 5000 Movies (Kaggle) |
| **Libraries** | Pandas, Scikit-learn, Matplotlib, Seaborn |

---

## How It Works

```
📊 Dataset → 🧹 Preprocessing → 🔧 Feature Engineering → 🔢 TF-IDF → 📐 Cosine Similarity → 🎯 Recommendations
```

---
## Step 1: Install Dependencies

In [ ]:
!pip install pandas scikit-learn matplotlib seaborn wordcloud -q
print("All dependencies installed!")

---
## Step 2: Download Dataset from Kaggle

We use the **TMDB 5000 Movie Dataset** — a real-world dataset with 4800+ movies including genres, cast, crew, keywords, and ratings.

In [ ]:
import os

# Method 1: Download using Kaggle API (recommended)
# Upload your kaggle.json or set credentials
try:
    os.makedirs('/root/.kaggle', exist_ok=True)
    # If kaggle.json is uploaded, copy it
    if os.path.exists('kaggle.json'):
        !cp kaggle.json /root/.kaggle/
        !chmod 600 /root/.kaggle/kaggle.json
    !pip install kaggle -q
    !kaggle datasets download -d tmdb/tmdb-movie-metadata -q
    !unzip -o tmdb-movie-metadata.zip -d data/
    print("Dataset downloaded from Kaggle!")
except:
    print("Kaggle API not configured. Using Method 2...")

# Method 2: Direct upload (fallback)
if not os.path.exists('data/tmdb_5000_movies.csv'):
    os.makedirs('data', exist_ok=True)
    from google.colab import files
    print("\n📁 Please upload these 2 files:")
    print("   1. tmdb_5000_movies.csv")
    print("   2. tmdb_5000_credits.csv")
    print("\n(Download from: https://www.kaggle.com/datasets/tmdb/tmdb-movie-metadata)\n")
    uploaded = files.upload()
    for filename in uploaded:
        os.rename(filename, f'data/{filename}')
    print("\nFiles uploaded successfully!")
else:
    print("Dataset already exists!")

print(f"\nFiles in data/: {os.listdir('data/')}")

---
## Step 3: Load & Explore the Dataset

In [ ]:
import pandas as pd
import numpy as np
import json
import ast
import warnings
warnings.filterwarnings('ignore')

# Load datasets
movies = pd.read_csv('data/tmdb_5000_movies.csv')
credits = pd.read_csv('data/tmdb_5000_credits.csv')

print(f"Movies dataset:  {movies.shape[0]} rows x {movies.shape[1]} columns")
print(f"Credits dataset: {credits.shape[0]} rows x {credits.shape[1]} columns")
print(f"\n--- Movies Columns ---")
print(movies.columns.tolist())
print(f"\n--- Credits Columns ---")
print(credits.columns.tolist())

In [ ]:
# Preview the movies dataset
movies[['title', 'genres', 'overview', 'vote_average', 'popularity']].head(10)

In [ ]:
# Dataset statistics
print("=" * 60)
print("         DATASET STATISTICS")
print("=" * 60)
print(f"Total Movies:        {len(movies)}")
print(f"Avg Rating:          {movies['vote_average'].mean():.2f}")
print(f"Avg Popularity:      {movies['popularity'].mean():.2f}")
print(f"Avg Runtime:         {movies['runtime'].mean():.0f} minutes")
print(f"Year Range:          {pd.to_datetime(movies['release_date'], errors='coerce').dt.year.min():.0f} - {pd.to_datetime(movies['release_date'], errors='coerce').dt.year.max():.0f}")
print(f"Missing Overviews:   {movies['overview'].isna().sum()}")
print(f"Missing Genres:      {movies['genres'].isna().sum()}")
print("=" * 60)

---
## Step 4: Data Preprocessing & Merging

In [ ]:
# Merge movies and credits on movie ID
if 'movie_id' in credits.columns:
    credits = credits.rename(columns={'movie_id': 'id'})

df = movies.merge(credits, on='id', how='left')

# Handle title columns after merge
if 'title_x' in df.columns:
    df['title'] = df['title_x']
    df = df.drop(columns=['title_x', 'title_y'], errors='ignore')

print(f"Merged dataset: {df.shape[0]} rows x {df.shape[1]} columns")

# Fill missing values
for col in ['overview', 'genres', 'keywords', 'cast', 'crew']:
    if col in df.columns:
        df[col] = df[col].fillna('')

print(f"Missing values after cleanup: {df[['overview','genres','keywords']].isna().sum().sum()}")
df[['title','genres','overview','vote_average']].head()

---
## Step 5: Feature Engineering

We extract structured features from JSON-like columns and combine them into a unified **feature "soup"** per movie.

```
Feature Soup = Genres + Keywords + Top-5 Cast + Director + Overview
```

In [ ]:
def safe_parse(text):
    """Parse JSON-like strings from CSV columns."""
    try:
        return json.loads(text.replace("'", '"')) if isinstance(text, str) else []
    except:
        try:
            return ast.literal_eval(text) if isinstance(text, str) else []
        except:
            return []

def extract_names(json_str, key='name', limit=None):
    """Extract name values from a JSON-like column."""
    parsed = safe_parse(json_str)
    names = [item.get(key, '') for item in parsed if isinstance(item, dict)]
    if limit:
        names = names[:limit]
    return ' '.join(names)

def extract_director(crew_str):
    """Extract director name from crew JSON."""
    parsed = safe_parse(crew_str)
    for member in parsed:
        if isinstance(member, dict) and member.get('job') == 'Director':
            return member.get('name', '')
    return ''

# Extract features
df['genre_list']   = df['genres'].apply(lambda x: [g['name'] for g in safe_parse(x) if isinstance(g, dict)])
df['genre_str']    = df['genres'].apply(lambda x: extract_names(x))
df['keyword_str']  = df['keywords'].apply(lambda x: extract_names(x))
df['cast_str']     = df['cast'].apply(lambda x: extract_names(x, limit=5))
df['director']     = df['crew'].apply(extract_director)
df['year']         = pd.to_datetime(df['release_date'], errors='coerce').dt.year.fillna(0).astype(int)

# Build the FEATURE SOUP
df['soup'] = (
    df['genre_str'] + ' ' +
    df['keyword_str'] + ' ' +
    df['cast_str'] + ' ' +
    df['director'] + ' ' +
    df['overview'].fillna('')
).str.lower()

# Drop duplicates
df = df.drop_duplicates(subset='title').reset_index(drop=True)

print(f"Total unique movies: {len(df)}")
print(f"\n--- Sample Feature Soup (Avatar) ---")
print(df[df['title'] == 'Avatar']['soup'].values[0][:300] + '...')

---
## Step 6: Visualize the Dataset

Let's explore the data with charts before building the recommendation engine.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Set dark theme for Hotstar vibes
plt.style.use('dark_background')
COLORS = ['#00d4ff', '#7b2ff7', '#ff2d55', '#ff9500', '#34c759', '#5856d6', '#ff375f', '#30d158']

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('HOTSTAR — Dataset Analysis', fontsize=20, fontweight='bold', color='#00d4ff', y=1.02)

# 1. Rating Distribution
axes[0,0].hist(df['vote_average'], bins=30, color='#7b2ff7', edgecolor='#1a1a2e', alpha=0.9)
axes[0,0].set_title('Rating Distribution', fontsize=14, fontweight='bold')
axes[0,0].set_xlabel('Vote Average')
axes[0,0].set_ylabel('Count')
axes[0,0].axvline(df['vote_average'].mean(), color='#ff2d55', linestyle='--', label=f'Mean: {df["vote_average"].mean():.1f}')
axes[0,0].legend()

# 2. Top 10 Genres
from collections import Counter
all_genres = [g for sublist in df['genre_list'] for g in sublist]
genre_counts = Counter(all_genres).most_common(10)
genres, counts = zip(*genre_counts)
bars = axes[0,1].barh(list(genres)[::-1], list(counts)[::-1], color=COLORS[:10])
axes[0,1].set_title('Top 10 Genres', fontsize=14, fontweight='bold')
axes[0,1].set_xlabel('Number of Movies')

# 3. Movies Released Per Year
year_counts = df[df['year'] > 1950].groupby('year').size()
axes[1,0].fill_between(year_counts.index, year_counts.values, alpha=0.3, color='#00d4ff')
axes[1,0].plot(year_counts.index, year_counts.values, color='#00d4ff', linewidth=2)
axes[1,0].set_title('Movies Released Per Year', fontsize=14, fontweight='bold')
axes[1,0].set_xlabel('Year')
axes[1,0].set_ylabel('Count')

# 4. Popularity vs Rating scatter
sample = df.nlargest(200, 'popularity')
scatter = axes[1,1].scatter(sample['vote_average'], sample['popularity'],
                            c=sample['vote_average'], cmap='cool', alpha=0.7, s=40, edgecolors='white', linewidth=0.3)
axes[1,1].set_title('Popularity vs Rating (Top 200)', fontsize=14, fontweight='bold')
axes[1,1].set_xlabel('Vote Average')
axes[1,1].set_ylabel('Popularity')
plt.colorbar(scatter, ax=axes[1,1], label='Rating')

plt.tight_layout()
plt.savefig('dataset_analysis.png', dpi=150, bbox_inches='tight', facecolor='#0a0c1a')
plt.show()
print("Charts saved!")

In [ ]:
# Word Cloud of movie overviews
from wordcloud import WordCloud

text = ' '.join(df['soup'].dropna().values)
wordcloud = WordCloud(
    width=1200, height=500,
    background_color='#0a0c1a',
    colormap='cool',
    max_words=150,
    contour_width=1,
    contour_color='#7b2ff7'
).generate(text)

fig, ax = plt.subplots(figsize=(16, 6))
ax.imshow(wordcloud, interpolation='bilinear')
ax.axis('off')
ax.set_title('Word Cloud — Movie Features', fontsize=18, fontweight='bold', color='#00d4ff', pad=20)
plt.tight_layout()
plt.savefig('wordcloud.png', dpi=150, bbox_inches='tight', facecolor='#0a0c1a')
plt.show()

---
## Step 7: TF-IDF Vectorization

**TF-IDF (Term Frequency — Inverse Document Frequency)** converts text features into numerical vectors.

- **TF** = How often a word appears in a movie's features
- **IDF** = How rare the word is across ALL movies
- **TF-IDF** = TF × IDF → Words that are *frequent in this movie but rare overall* get higher scores

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Build TF-IDF matrix
tfidf = TfidfVectorizer(
    stop_words='english',   # Remove common words (the, is, at, etc.)
    max_features=15000      # Top 15,000 most important terms
)

tfidf_matrix = tfidf.fit_transform(df['soup'])

print(f"TF-IDF Matrix Shape: {tfidf_matrix.shape}")
print(f"  → {tfidf_matrix.shape[0]} movies")
print(f"  → {tfidf_matrix.shape[1]} unique features (words)")
print(f"  → Matrix density: {tfidf_matrix.nnz / (tfidf_matrix.shape[0] * tfidf_matrix.shape[1]) * 100:.2f}%")
print(f"\nSample feature names: {tfidf.get_feature_names_out()[:20].tolist()}")

In [ ]:
# Visualize TF-IDF scores for a sample movie
avatar_idx = df[df['title'] == 'Avatar'].index[0]
avatar_tfidf = tfidf_matrix[avatar_idx].toarray().flatten()

# Get top features for Avatar
feature_names = tfidf.get_feature_names_out()
top_indices = avatar_tfidf.argsort()[-15:][::-1]
top_features = [(feature_names[i], avatar_tfidf[i]) for i in top_indices]

fig, ax = plt.subplots(figsize=(12, 5))
features, scores = zip(*top_features)
bars = ax.barh(list(features)[::-1], list(scores)[::-1],
               color=['#00d4ff', '#1aa3cc', '#7b2ff7', '#9b59f7', '#ff2d55',
                      '#ff5c7a', '#ff9500', '#ffaa33', '#34c759', '#5dd67a',
                      '#5856d6', '#7a78e6', '#00d4ff', '#1aa3cc', '#7b2ff7'])
ax.set_title('Top 15 TF-IDF Features for "Avatar"', fontsize=16, fontweight='bold', color='#00d4ff')
ax.set_xlabel('TF-IDF Score', fontsize=12)
plt.tight_layout()
plt.savefig('tfidf_avatar.png', dpi=150, bbox_inches='tight', facecolor='#0a0c1a')
plt.show()

---
## Step 8: Cosine Similarity Matrix

**Cosine Similarity** measures the angle between two movie vectors. A score of:
- **1.0** = Identical features (same movie)
- **0.0** = Completely different
- **0.5+** = Very similar movies

$$\text{cosine\_similarity}(A, B) = \frac{A \cdot B}{\|A\| \times \|B\|}$$

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

# Compute the cosine similarity matrix
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

print(f"Cosine Similarity Matrix Shape: {cosine_sim.shape}")
print(f"  → {cosine_sim.shape[0]} x {cosine_sim.shape[1]} pairwise comparisons")
print(f"  → Total comparisons: {cosine_sim.shape[0] * cosine_sim.shape[1]:,}")
print(f"\nDiagonal (self-similarity): {cosine_sim[0][0]:.4f} (should be 1.0)")
print(f"Average similarity: {cosine_sim.mean():.6f}")

In [ ]:
# Visualize Cosine Similarity Heatmap (top 20 movies)
top_movies = df.nlargest(20, 'popularity')
top_indices = top_movies.index.tolist()
top_titles = top_movies['title'].tolist()

# Extract sub-matrix
sim_subset = cosine_sim[np.ix_(top_indices, top_indices)]

fig, ax = plt.subplots(figsize=(14, 11))
sns.heatmap(sim_subset,
            xticklabels=top_titles,
            yticklabels=top_titles,
            cmap='cool',
            annot=True,
            fmt='.2f',
            annot_kws={'size': 7},
            linewidths=0.5,
            linecolor='#1a1a2e',
            cbar_kws={'label': 'Cosine Similarity'},
            ax=ax)
ax.set_title('Cosine Similarity Heatmap — Top 20 Popular Movies',
             fontsize=16, fontweight='bold', color='#00d4ff', pad=20)
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.yticks(rotation=0, fontsize=8)
plt.tight_layout()
plt.savefig('cosine_heatmap.png', dpi=150, bbox_inches='tight', facecolor='#0a0c1a')
plt.show()

---
## Step 9: Build the Recommendation Function

In [ ]:
# Create title-to-index mapping
title_to_index = {title.lower(): idx for idx, title in enumerate(df['title'])}

def recommend(title, n=10):
    """
    Get top-N movie recommendations based on content similarity.
    
    Parameters:
        title (str): Movie title to find similar movies for
        n (int): Number of recommendations to return
        
    Returns:
        DataFrame with recommended movies and similarity scores
    """
    idx = title_to_index.get(title.lower())
    if idx is None:
        print(f"Movie '{title}' not found! Try a different title.")
        return None
    
    # Get pairwise similarity scores
    sim_scores = list(enumerate(cosine_sim[idx]))
    
    # Sort by similarity (descending)
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    
    # Skip the first one (itself) and get top N
    sim_scores = sim_scores[1:n+1]
    
    # Build results
    results = []
    for rank, (movie_idx, score) in enumerate(sim_scores, 1):
        row = df.iloc[movie_idx]
        results.append({
            'Rank': rank,
            'Title': row['title'],
            'Similarity': f'{score*100:.1f}%',
            'Genres': ', '.join(row['genre_list'][:3]),
            'Rating': f"{row['vote_average']:.1f}",
            'Year': int(row['year']) if row['year'] > 0 else 'N/A',
            'Director': row['director']
        })
    
    return pd.DataFrame(results)

print("Recommendation function created!")
print(f"Total movies in database: {len(title_to_index)}")

---
## Step 10: Test the Recommendations! 🎬

Let's test our recommendation engine with some popular movies.

In [ ]:
# Test 1: Avatar
print("=" * 70)
print("  RECOMMENDATIONS FOR: AVATAR")
print("=" * 70)
recommend('Avatar')

In [ ]:
# Test 2: The Dark Knight
print("=" * 70)
print("  RECOMMENDATIONS FOR: THE DARK KNIGHT")
print("=" * 70)
recommend('The Dark Knight')

In [ ]:
# Test 3: Inception
print("=" * 70)
print("  RECOMMENDATIONS FOR: INCEPTION")
print("=" * 70)
recommend('Inception')

In [ ]:
# Test 4: Titanic
print("=" * 70)
print("  RECOMMENDATIONS FOR: TITANIC")
print("=" * 70)
recommend('Titanic')

---
## Step 11: Visualize Recommendations

In [ ]:
def visualize_recommendations(title, n=10):
    """
    Visualize movie recommendations as a horizontal bar chart.
    """
    idx = title_to_index.get(title.lower())
    if idx is None:
        print(f"Movie '{title}' not found!")
        return
    
    sim_scores = sorted(enumerate(cosine_sim[idx]), key=lambda x: x[1], reverse=True)[1:n+1]
    titles = [df.iloc[i]['title'] for i, _ in sim_scores]
    scores = [s * 100 for _, s in sim_scores]
    
    fig, ax = plt.subplots(figsize=(12, 6))
    
    # Create gradient colors
    colors = plt.cm.cool(np.linspace(0.2, 0.9, len(titles)))
    
    bars = ax.barh(titles[::-1], scores[::-1], color=colors[::-1], edgecolor='#1a1a2e', height=0.7)
    
    # Add percentage labels
    for bar, score in zip(bars, scores[::-1]):
        ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
                f'{score:.1f}%', va='center', fontsize=11, fontweight='bold', color='#00d4ff')
    
    ax.set_title(f'Top {n} Recommendations for "{title}"',
                 fontsize=16, fontweight='bold', color='#ff2d55', pad=15)
    ax.set_xlabel('Similarity Score (%)', fontsize=12)
    ax.set_xlim(0, max(scores) + 8)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    
    plt.tight_layout()
    plt.savefig(f'recommendations_{title.lower().replace(" ","_")}.png', dpi=150, bbox_inches='tight', facecolor='#0a0c1a')
    plt.show()

# Visualize recommendations for multiple movies
visualize_recommendations('Avatar')
visualize_recommendations('Interstellar')
visualize_recommendations('The Avengers')

---
## Step 12: Interactive Recommendation Widget

An interactive dropdown to select a movie and get instant recommendations!

In [ ]:
from IPython.display import display, HTML, clear_output
import ipywidgets as widgets

# Get sorted list of all movie titles
all_titles = sorted(df['title'].tolist())

# Create interactive widget
movie_dropdown = widgets.Combobox(
    placeholder='Type a movie name...',
    options=all_titles,
    description='Movie:',
    ensure_option=False,
    layout=widgets.Layout(width='500px')
)

n_slider = widgets.IntSlider(
    value=10, min=5, max=20, step=1,
    description='Top N:',
    style={'description_width': 'initial'}
)

output = widgets.Output()

def on_recommend(change):
    with output:
        clear_output(wait=True)
        title = movie_dropdown.value
        if title:
            result = recommend(title, n_slider.value)
            if result is not None:
                print(f"\n{'='*70}")
                print(f"  RECOMMENDATIONS FOR: {title.upper()}")
                print(f"{'='*70}\n")
                display(result.style.set_properties(**{
                    'background-color': '#0a0c1a',
                    'color': '#f0f2ff',
                    'border-color': '#1a1a2e'
                }))
                print()
                visualize_recommendations(title, n_slider.value)

btn = widgets.Button(description='Get Recommendations', button_style='info',
                     layout=widgets.Layout(width='200px'))
btn.on_click(lambda b: on_recommend(None))

display(widgets.VBox([
    widgets.HTML('<h3 style="color:#00d4ff;">🎬 Interactive Movie Recommender</h3>'),
    widgets.HBox([movie_dropdown, n_slider, btn]),
    output
]))

---
## Step 13: Full 3D Interactive Frontend (Inline)

Below is the complete **Hotstar-themed 3D frontend** rendered directly in the notebook using Three.js and GSAP!

In [ ]:
# Prepare movie data as JSON for the frontend
import json

def movies_to_json(dataframe, n=20, sort_by='popularity'):
    """Convert movies DataFrame to JSON for frontend."""
    top = dataframe.nlargest(n, sort_by)
    result = []
    for _, row in top.iterrows():
        result.append({
            'title': str(row['title']),
            'genres': row['genre_list'][:3] if isinstance(row['genre_list'], list) else [],
            'vote_average': float(row.get('vote_average', 0)),
            'year': int(row.get('year', 0)),
            'overview': str(row.get('overview', ''))[:200],
            'runtime': int(row.get('runtime', 0)) if pd.notna(row.get('runtime')) else 0,
            'director': str(row.get('director', '')),
        })
    return result

# Recommendation function that returns JSON
def recommend_json(title, n=10):
    idx = title_to_index.get(title.lower())
    if idx is None:
        return []
    sim_scores = sorted(enumerate(cosine_sim[idx]), key=lambda x: x[1], reverse=True)[1:n+1]
    results = []
    for movie_idx, score in sim_scores:
        row = df.iloc[movie_idx]
        results.append({
            'title': str(row['title']),
            'genres': row['genre_list'][:3] if isinstance(row['genre_list'], list) else [],
            'vote_average': float(row.get('vote_average', 0)),
            'year': int(row.get('year', 0)),
            'similarity': round(score * 100, 1),
            'director': str(row.get('director', '')),
            'overview': str(row.get('overview', ''))[:200],
            'runtime': int(row.get('runtime', 0)) if pd.notna(row.get('runtime')) else 0,
        })
    return results

# Pre-compute data for embedded frontend
trending_json = json.dumps(movies_to_json(df, 15, 'popularity'))
toprated_json = json.dumps(movies_to_json(df[df['vote_count'] >= 100], 15, 'vote_average'))
all_titles_json = json.dumps(sorted(df['title'].tolist()))

# Pre-compute recommendations for several movies
precomputed = {}
for t in ['Avatar', 'The Dark Knight', 'Inception', 'Titanic', 'Interstellar',
          'The Avengers', 'Jurassic World', 'Frozen', 'The Lion King', 'Toy Story',
          'Iron Man', 'Spider-Man', 'Harry Potter and the Philosopher\'s Stone',
          'The Matrix', 'Forrest Gump', 'Gladiator', 'The Godfather', 'Pulp Fiction',
          'Fight Club', 'The Shawshank Redemption', 'Mad Max: Fury Road']:
    recs = recommend_json(t, 12)
    if recs:
        precomputed[t.lower()] = recs

precomputed_json = json.dumps(precomputed)

print(f"Prepared {len(json.loads(trending_json))} trending movies")
print(f"Prepared {len(json.loads(toprated_json))} top-rated movies")
print(f"Pre-computed recommendations for {len(precomputed)} movies")
print(f"Total searchable titles: {len(json.loads(all_titles_json))}")

In [ ]:
from IPython.display import HTML

frontend_html = f"""
<div id="hotstar-app" style="font-family:'Inter',sans-serif;background:#05060f;color:#f0f2ff;border-radius:16px;overflow:hidden;position:relative;">
  <style>
    @import url('https://fonts.googleapis.com/css2?family=Inter:wght@300;400;500;600;700;800;900&family=Outfit:wght@300;400;500;600;700;800;900&display=swap');
    #hotstar-app * {{ box-sizing: border-box; margin: 0; padding: 0; }}
    #hotstar-app .hs-hero {{ position:relative;padding:60px 40px;text-align:center;overflow:hidden;min-height:400px; }}
    #hotstar-app .hs-hero canvas {{ position:absolute;top:0;left:0;width:100%;height:100%;z-index:0; }}
    #hotstar-app .hs-hero-content {{ position:relative;z-index:2; }}
    #hotstar-app .hs-badge {{ display:inline-block;background:rgba(255,255,255,0.06);border:1px solid rgba(255,255,255,0.1);padding:6px 16px;border-radius:100px;font-size:12px;color:#00d4ff;margin-bottom:16px; }}
    #hotstar-app .hs-title {{ font-family:'Outfit',sans-serif;font-size:42px;font-weight:900;line-height:1.1;margin-bottom:16px; }}
    #hotstar-app .hs-gradient {{ background:linear-gradient(135deg,#00d4ff,#7b2ff7,#ff2d55);-webkit-background-clip:text;-webkit-text-fill-color:transparent;background-clip:text; }}
    #hotstar-app .hs-subtitle {{ font-size:15px;color:rgba(240,242,255,0.6);max-width:600px;margin:0 auto 24px;line-height:1.6; }}
    #hotstar-app .hs-section {{ padding:40px 30px; }}
    #hotstar-app .hs-section-title {{ font-family:'Outfit',sans-serif;font-size:24px;font-weight:800;margin-bottom:8px;text-align:center; }}
    #hotstar-app .hs-section-desc {{ font-size:13px;color:rgba(240,242,255,0.5);text-align:center;margin-bottom:24px; }}
    #hotstar-app .hs-tag {{ display:inline-block;background:rgba(255,255,255,0.04);border:1px solid rgba(255,255,255,0.06);padding:4px 14px;border-radius:100px;font-size:12px;color:#00d4ff;margin-bottom:12px; }}
    #hotstar-app .hs-carousel {{ display:flex;gap:16px;overflow-x:auto;padding:10px 0 20px;scrollbar-width:none; }}
    #hotstar-app .hs-carousel::-webkit-scrollbar {{ display:none; }}
    #hotstar-app .hs-card {{ flex-shrink:0;width:180px;border-radius:14px;background:rgba(15,18,40,0.7);border:1px solid rgba(255,255,255,0.06);overflow:hidden;cursor:pointer;transition:all 0.4s cubic-bezier(0.4,0,0.2,1); }}
    #hotstar-app .hs-card:hover {{ transform:translateY(-8px) scale(1.02);border-color:#7b2ff7;box-shadow:0 15px 40px rgba(123,47,247,0.2); }}
    #hotstar-app .hs-poster {{ height:200px;display:flex;align-items:center;justify-content:center;font-size:48px;position:relative; }}
    #hotstar-app .hs-poster::after {{ content:'';position:absolute;bottom:0;left:0;right:0;height:50%;background:linear-gradient(to top,rgba(15,18,40,0.9),transparent); }}
    #hotstar-app .hs-rating {{ position:absolute;top:8px;right:8px;background:rgba(0,0,0,0.7);padding:3px 8px;border-radius:6px;font-size:11px;font-weight:700;color:#ff9500;z-index:2; }}
    #hotstar-app .hs-match {{ position:absolute;top:8px;left:8px;background:linear-gradient(135deg,#7b2ff7,#ff2d55);padding:3px 8px;border-radius:6px;font-size:10px;font-weight:700;color:white;z-index:2; }}
    #hotstar-app .hs-card-info {{ padding:12px; }}
    #hotstar-app .hs-card-title {{ font-family:'Outfit',sans-serif;font-size:13px;font-weight:700;white-space:nowrap;overflow:hidden;text-overflow:ellipsis; }}
    #hotstar-app .hs-card-meta {{ font-size:11px;color:rgba(240,242,255,0.4);margin-top:4px; }}
    #hotstar-app .hs-card-genres {{ display:flex;flex-wrap:wrap;gap:4px;margin-top:6px; }}
    #hotstar-app .hs-genre-tag {{ font-size:9px;padding:2px 7px;border-radius:4px;background:rgba(255,255,255,0.04);border:1px solid rgba(255,255,255,0.08);color:rgba(240,242,255,0.6); }}
    #hotstar-app .hs-rec-box {{ max-width:600px;margin:0 auto 30px;display:flex;gap:8px; }}
    #hotstar-app .hs-rec-input {{ flex:1;background:rgba(15,18,40,0.7);border:2px solid rgba(255,255,255,0.08);border-radius:14px;padding:12px 18px;color:#f0f2ff;font-size:14px;outline:none;font-family:'Inter',sans-serif;transition:border-color 0.3s; }}
    #hotstar-app .hs-rec-input:focus {{ border-color:#7b2ff7;box-shadow:0 0 20px rgba(123,47,247,0.2); }}
    #hotstar-app .hs-rec-btn {{ background:linear-gradient(135deg,#7b2ff7,#ff2d55);color:white;border:none;padding:12px 24px;border-radius:14px;font-size:14px;font-weight:700;cursor:pointer;font-family:'Inter',sans-serif;white-space:nowrap;transition:all 0.3s; }}
    #hotstar-app .hs-rec-btn:hover {{ box-shadow:0 4px 20px rgba(123,47,247,0.4);transform:translateY(-1px); }}
    #hotstar-app .hs-rec-grid {{ display:grid;grid-template-columns:repeat(auto-fill,minmax(180px,1fr));gap:16px; }}
    #hotstar-app .hs-pipeline {{ display:flex;justify-content:center;flex-wrap:wrap;gap:8px;margin:20px 0; }}
    #hotstar-app .hs-step {{ background:rgba(15,18,40,0.7);border:1px solid rgba(255,255,255,0.06);border-radius:12px;padding:16px 12px;text-align:center;width:140px;transition:all 0.3s; }}
    #hotstar-app .hs-step:hover {{ border-color:#00d4ff;box-shadow:0 0 20px rgba(0,212,255,0.15);transform:translateY(-4px); }}
    #hotstar-app .hs-step-icon {{ font-size:28px;margin-bottom:8px; }}
    #hotstar-app .hs-step h4 {{ font-size:12px;font-weight:700;margin-bottom:4px; }}
    #hotstar-app .hs-step p {{ font-size:10px;color:rgba(240,242,255,0.4);line-height:1.4; }}
    #hotstar-app .hs-arrow {{ display:flex;align-items:center;color:rgba(123,47,247,0.5);font-size:20px; }}
    #hotstar-app .hs-stats {{ display:flex;justify-content:center;gap:40px;margin-top:20px; }}
    #hotstar-app .hs-stat-num {{ font-family:'Outfit',sans-serif;font-size:28px;font-weight:800;background:linear-gradient(135deg,#00d4ff,#7b2ff7,#ff2d55);-webkit-background-clip:text;-webkit-text-fill-color:transparent; }}
    #hotstar-app .hs-stat-label {{ font-size:11px;color:rgba(240,242,255,0.3);text-transform:uppercase;letter-spacing:1px; }}
    #hotstar-app .hs-suggestions {{ max-width:600px;margin:-20px auto 20px;background:rgba(10,12,26,0.95);border:1px solid rgba(255,255,255,0.08);border-radius:12px;max-height:200px;overflow-y:auto;display:none; }}
    #hotstar-app .hs-sug-item {{ padding:10px 18px;cursor:pointer;font-size:13px;border-bottom:1px solid rgba(255,255,255,0.04);transition:background 0.2s;display:flex;justify-content:space-between; }}
    #hotstar-app .hs-sug-item:hover {{ background:rgba(255,255,255,0.06); }}
    #hotstar-app .hs-sug-item:last-child {{ border-bottom:none; }}
    #hotstar-app .hs-sug-year {{ color:rgba(240,242,255,0.3);font-size:12px; }}
    #hotstar-app .hs-empty {{ text-align:center;padding:40px;color:rgba(240,242,255,0.3);font-size:14px; }}
    #hotstar-app .hs-footer {{ text-align:center;padding:20px;border-top:1px solid rgba(255,255,255,0.06);font-size:11px;color:rgba(240,242,255,0.25); }}
  </style>

  <!-- HERO -->
  <div class="hs-hero">
    <canvas id="hs-bg-canvas"></canvas>
    <div class="hs-hero-content">
      <div class="hs-badge">AI-Powered Recommendations</div>
      <h1 class="hs-title">Discover Movies<br><span class="hs-gradient">You'll Love</span></h1>
      <p class="hs-subtitle">Intelligent movie recommendations powered by TF-IDF vectorization and cosine similarity on the TMDB 5000 dataset.</p>
      <div class="hs-stats">
        <div><div class="hs-stat-num">4800</div><div class="hs-stat-label">Movies</div></div>
        <div><div class="hs-stat-num">20</div><div class="hs-stat-label">Genres</div></div>
        <div><div class="hs-stat-num">15K</div><div class="hs-stat-label">Features</div></div>
      </div>
    </div>
  </div>

  <!-- TRENDING -->
  <div class="hs-section">
    <div style="text-align:center;"><span class="hs-tag">Trending Now</span></div>
    <h2 class="hs-section-title">Most Popular Movies</h2>
    <p class="hs-section-desc">Top movies by popularity score from TMDB dataset</p>
    <div class="hs-carousel" id="hs-trending"></div>
  </div>

  <!-- TOP RATED -->
  <div class="hs-section">
    <div style="text-align:center;"><span class="hs-tag">Top Rated</span></div>
    <h2 class="hs-section-title">Highest Rated Movies</h2>
    <p class="hs-section-desc">Highest rated movies (minimum 100 votes)</p>
    <div class="hs-carousel" id="hs-toprated"></div>
  </div>

  <!-- RECOMMENDATION ENGINE -->
  <div class="hs-section" style="background:rgba(123,47,247,0.03);border-top:1px solid rgba(255,255,255,0.04);border-bottom:1px solid rgba(255,255,255,0.04);">
    <div style="text-align:center;"><span class="hs-tag">AI Engine</span></div>
    <h2 class="hs-section-title">Get Recommendations</h2>
    <p class="hs-section-desc">Type a movie you love and our ML engine finds similar ones</p>
    <div class="hs-rec-box">
      <input type="text" class="hs-rec-input" id="hs-rec-input" placeholder="Try: Avatar, Inception, The Dark Knight..." autocomplete="off">
      <button class="hs-rec-btn" onclick="hsRecommend()">Recommend</button>
    </div>
    <div class="hs-suggestions" id="hs-suggestions"></div>
    <div id="hs-rec-results">
      <div class="hs-empty">Enter a movie title above to get AI-powered recommendations</div>
    </div>
  </div>

  <!-- HOW IT WORKS -->
  <div class="hs-section">
    <div style="text-align:center;"><span class="hs-tag">Under the Hood</span></div>
    <h2 class="hs-section-title">How It Works</h2>
    <p class="hs-section-desc">Our ML-powered content-based filtering pipeline</p>
    <div class="hs-pipeline">
      <div class="hs-step"><div class="hs-step-icon">📊</div><h4>Data Collection</h4><p>TMDB 5000 dataset from Kaggle</p></div>
      <div class="hs-arrow">→</div>
      <div class="hs-step"><div class="hs-step-icon">🔧</div><h4>Feature Engineering</h4><p>Genres + Keywords + Cast + Director</p></div>
      <div class="hs-arrow">→</div>
      <div class="hs-step"><div class="hs-step-icon">🔢</div><h4>TF-IDF</h4><p>15,000-dim text vectors</p></div>
      <div class="hs-arrow">→</div>
      <div class="hs-step"><div class="hs-step-icon">📐</div><h4>Cosine Similarity</h4><p>Pairwise similarity matrix</p></div>
      <div class="hs-arrow">→</div>
      <div class="hs-step"><div class="hs-step-icon">🎯</div><h4>Recommendations</h4><p>Top-N ranked results</p></div>
    </div>
  </div>

  <div class="hs-footer">
    HOTSTAR AI — Data Driven Recommendations | TMDB 5000 Dataset | Content-Based Filtering (TF-IDF + Cosine Similarity)
  </div>
</div>

<script src="https://cdnjs.cloudflare.com/ajax/libs/three.js/r128/three.min.js"></script>
<script>
(function() {{
  // Data
  const trending = {trending_json};
  const toprated = {toprated_json};
  const allTitles = {all_titles_json};
  const precomputed = {precomputed_json};

  const GENRE_EMOJIS = {{
    'Action':'💥','Adventure':'🗺️','Animation':'🎨','Comedy':'😂','Crime':'🔪',
    'Documentary':'📹','Drama':'🎭','Family':'👨‍👩‍👧‍👦','Fantasy':'🧙','History':'📜',
    'Horror':'👻','Music':'🎵','Mystery':'🔍','Romance':'💕','Science Fiction':'🚀',
    'TV Movie':'📺','Thriller':'😰','War':'⚔️','Western':'🤠'
  }};

  const GRADS = [
    ['#1a0533','#4a148c'],['#0d2137','#1565c0'],['#2d0a1e','#880e4f'],
    ['#0a2d1a','#1b5e20'],['#2d2a0a','#f57f17'],['#1a0a2d','#6a1b9a'],
    ['#2d0a0a','#b71c1c'],['#0a1a2d','#0d47a1'],['#0d2d2d','#00695c'],
    ['#2d1a0a','#e65100'],['#1a102d','#4527a0'],['#2d0d1a','#ad1457']
  ];

  function getEmoji(genres) {{
    if (!genres || !genres.length) return '🎬';
    for (const g of genres) if (GENRE_EMOJIS[g]) return GENRE_EMOJIS[g];
    return '🎬';
  }}

  function createCard(m, i, showMatch) {{
    const g = GRADS[i % GRADS.length];
    const emoji = getEmoji(m.genres);
    const genreTags = (m.genres||[]).slice(0,2).map(g => `<span class="hs-genre-tag">${{g}}</span>`).join('');
    return `<div class="hs-card">
      <div class="hs-poster" style="background:linear-gradient(135deg,${{g[0]}},${{g[1]}})">
        ${{emoji}}
        <div class="hs-rating">⭐ ${{m.vote_average?.toFixed(1)||'N/A'}}</div>
        ${{showMatch && m.similarity ? `<div class="hs-match">${{m.similarity}}% Match</div>` : ''}}
      </div>
      <div class="hs-card-info">
        <div class="hs-card-title" title="${{m.title}}">${{m.title}}</div>
        <div class="hs-card-meta">${{m.year||''}} ${{m.runtime ? '· '+m.runtime+'m' : ''}}</div>
        <div class="hs-card-genres">${{genreTags}}</div>
      </div>
    </div>`;
  }}

  // Render carousels
  document.getElementById('hs-trending').innerHTML = trending.map((m,i) => createCard(m,i,false)).join('');
  document.getElementById('hs-toprated').innerHTML = toprated.map((m,i) => createCard(m,i+5,false)).join('');

  // Recommendation logic
  const input = document.getElementById('hs-rec-input');
  const suggestions = document.getElementById('hs-suggestions');

  input.addEventListener('input', function() {{
    const q = this.value.toLowerCase().trim();
    if (q.length < 2) {{ suggestions.style.display = 'none'; return; }}
    const matches = allTitles.filter(t => t.toLowerCase().includes(q)).slice(0, 8);
    if (matches.length) {{
      suggestions.innerHTML = matches.map(t => {{
        const movie = [...trending, ...toprated].find(m => m.title === t);
        return `<div class="hs-sug-item" onclick="document.getElementById('hs-rec-input').value='${{t.replace(/'/g,"\\'")}}';hsRecommend()">
          <span>${{t}}</span><span class="hs-sug-year">${{movie?.year||''}}</span></div>`;
      }}).join('');
      suggestions.style.display = 'block';
    }} else suggestions.style.display = 'none';
  }});

  input.addEventListener('keydown', function(e) {{
    if (e.key === 'Enter') {{ suggestions.style.display='none'; hsRecommend(); }}
  }});

  window.hsRecommend = function() {{
    const title = input.value.trim();
    suggestions.style.display = 'none';
    const results = document.getElementById('hs-rec-results');
    if (!title) return;

    const recs = precomputed[title.toLowerCase()];
    if (!recs || !recs.length) {{
      // Try fuzzy match
      const key = Object.keys(precomputed).find(k => k.includes(title.toLowerCase()));
      const fuzzyRecs = key ? precomputed[key] : null;
      if (fuzzyRecs) {{
        results.innerHTML = '<div class="hs-rec-grid">' + fuzzyRecs.map((m,i) => createCard(m,i,true)).join('') + '</div>';
      }} else {{
        results.innerHTML = `<div class="hs-empty">Movie "${{title}}" not found in pre-computed recommendations.<br>Try: Avatar, Inception, The Dark Knight, Interstellar, The Avengers, Titanic, The Matrix, etc.</div>`;
      }}
      return;
    }}
    results.innerHTML = '<div class="hs-rec-grid">' + recs.map((m,i) => createCard(m,i,true)).join('') + '</div>';
  }};

  // Three.js starfield background
  try {{
    const canvas = document.getElementById('hs-bg-canvas');
    const scene = new THREE.Scene();
    const camera = new THREE.PerspectiveCamera(75, canvas.clientWidth/canvas.clientHeight, 0.1, 1000);
    camera.position.z = 400;
    const renderer = new THREE.WebGLRenderer({{ canvas, alpha:true, antialias:true }});
    renderer.setSize(canvas.clientWidth, canvas.clientHeight);
    renderer.setPixelRatio(Math.min(window.devicePixelRatio, 2));

    const geo = new THREE.BufferGeometry();
    const count = 3000;
    const pos = new Float32Array(count*3);
    const cols = new Float32Array(count*3);
    const palette = [new THREE.Color('#00d4ff'),new THREE.Color('#7b2ff7'),new THREE.Color('#ff2d55'),new THREE.Color('#ffffff')];
    for(let i=0;i<count;i++) {{
      pos[i*3]=(Math.random()-0.5)*1200; pos[i*3+1]=(Math.random()-0.5)*800; pos[i*3+2]=(Math.random()-0.5)*600;
      const c=palette[Math.floor(Math.random()*palette.length)];
      cols[i*3]=c.r; cols[i*3+1]=c.g; cols[i*3+2]=c.b;
    }}
    geo.setAttribute('position', new THREE.BufferAttribute(pos,3));
    geo.setAttribute('color', new THREE.BufferAttribute(cols,3));
    const stars = new THREE.Points(geo, new THREE.PointsMaterial({{size:2,vertexColors:true,transparent:true,opacity:0.7}}));
    scene.add(stars);

    function animate() {{ requestAnimationFrame(animate); const t=Date.now()*0.00003; stars.rotation.y=t; stars.rotation.x=t*0.5; renderer.render(scene,camera); }}
    animate();
  }} catch(e) {{ console.log('Three.js not loaded:', e); }}
}})();
</script>
"""

display(HTML(frontend_html))

---
## Summary

### What We Built

| Component | Details |
|---|---|
| **Dataset** | TMDB 5000 Movies from Kaggle (4800 unique movies) |
| **Algorithm** | Content-Based Filtering |
| **Feature Engineering** | Genres + Keywords + Top-5 Cast + Director + Overview |
| **Vectorization** | TF-IDF (15,000-dim feature vectors) |
| **Similarity** | Cosine Similarity (4800 × 4800 matrix) |
| **Frontend** | 3D Hotstar-themed interface with Three.js |

### Key Results
- Content-based filtering successfully identifies similar movies based on shared features
- TF-IDF captures the importance of terms relative to the entire corpus
- Cosine similarity effectively measures multi-dimensional content alignment

### Libraries Used
- **pandas** — Data manipulation
- **scikit-learn** — TF-IDF Vectorization + Cosine Similarity
- **matplotlib / seaborn** — Data visualization
- **wordcloud** — Feature word cloud
- **Three.js** — 3D starfield background

---
*Data Driven Recommendations — Mini Project | Disney+ Hotstar Platform*